# Transcripción de VODs con Whisper (GPU T4 gratis)

**Antes de empezar:** `Runtime > Change runtime type > T4 GPU`

Ejecutar las celdas en orden: 1 → 2 → 3 → 4 → 5 → 6 (o 7 para todos los VODs).

In [ ]:
# Celda 1 — Instalar dependencias
!pip install faster-whisper yt-dlp tqdm requests -q

In [ ]:
# Celda 2 — Verificar GPU
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU — ve a Runtime > Change runtime type > T4 GPU")

In [ ]:
# Celda 3 — Configuración (editar aquí)
VOD_ID     = "2709649424"   # ← Cambiar por el VOD a transcribir
MODEL_NAME = "turbo"         # tiny/base/small/medium/large/turbo
API_BASE   = "https://backend.permisossubtel.cl/api"

In [ ]:
# Celda 4 — Código core
import json
import subprocess
import tempfile
import shutil
import time
import requests
from pathlib import Path
from tqdm import tqdm

UPLOAD_URL = f"{API_BASE}/transcripts/upload/"
VIDEOS_URL = f"{API_BASE}/videos/"
CACHE_DIR  = Path("/content/transcripts_cache")

WHISPER_SPEED_CPU = {
    "tiny": 80.0, "tiny.en": 80.0,
    "base": 40.0, "base.en": 40.0,
    "small": 15.0, "small.en": 15.0,
    "medium": 5.0, "medium.en": 5.0,
    "large-v1": 2.5, "large-v2": 2.5, "large-v3": 2.5, "large": 2.5,
    "large-v3-turbo": 20.0, "turbo": 20.0,
}
WHISPER_SPEED_GPU = {
    "tiny": 200.0, "tiny.en": 200.0,
    "base": 150.0, "base.en": 150.0,
    "small": 80.0, "small.en": 80.0,
    "medium": 35.0, "medium.en": 35.0,
    "large-v1": 15.0, "large-v2": 15.0, "large-v3": 15.0, "large": 15.0,
    "large-v3-turbo": 70.0, "turbo": 70.0,
}
DOWNLOAD_SPEED_X = 100.0


def fmt_seconds(s: float) -> str:
    s = int(s)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    parts = []
    if h: parts.append(f"{h}h")
    if m: parts.append(f"{m}m")
    parts.append(f"{sec}s")
    return " ".join(parts)


def get_video_info(vod_id: str) -> dict | None:
    try:
        resp = requests.get(f"{VIDEOS_URL}{vod_id}/", timeout=15)
        if resp.status_code == 200:
            return resp.json()
    except Exception:
        pass
    return None


def get_all_vods() -> list:
    vods, url = [], VIDEOS_URL
    while url:
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"[ERROR] No se pudo obtener VODs: {e}")
            return vods
        if isinstance(data, list):
            vods.extend(data)
            break
        vods.extend(data.get("results", []))
        url = data.get("next")
    return vods


def download_audio(vod_id: str, dest: Path, duration_s) -> bool:
    twitch_url = f"https://www.twitch.tv/videos/{vod_id}"
    if duration_s:
        est = duration_s / DOWNLOAD_SPEED_X
        print(f"  → Descargando audio  (duración VOD: {fmt_seconds(duration_s)}  |  estimado: ~{fmt_seconds(est)})")
    else:
        print(f"  → Descargando audio ({twitch_url})...")

    cmd = [
        "yt-dlp", "--no-playlist",
        "--format", "audio_only/bestaudio",
        "--concurrent-fragments", "16",
        "--progress", "--newline",
        "-o", f"{dest}.%(ext)s",
        twitch_url,
    ]
    t0 = time.time()
    result = subprocess.run(cmd)
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f"  ✗ yt-dlp falló ({fmt_seconds(elapsed)})")
        return False
    print(f"  ✓ Descarga completada en {fmt_seconds(elapsed)}")
    return True


def find_audio_file(base: Path) -> Path | None:
    for ext in ("mp4", "ts", "aac", "mp3", "m4a", "opus", "webm", "ogg", "wav"):
        p = base.parent / f"{base.name}.{ext}"
        if p.exists():
            return p
    return None


def transcribe(audio_path: Path, model, model_name: str, duration_s, use_gpu: bool = False) -> list:
    speed_table = WHISPER_SPEED_GPU if use_gpu else WHISPER_SPEED_CPU
    speed_x = speed_table.get(model_name, 20.0 if use_gpu else 5.0)
    est_seconds = int(duration_s / speed_x) if duration_s else None

    if est_seconds:
        print(f"  → Transcribiendo con faster-whisper '{model_name}'  (estimado: ~{fmt_seconds(est_seconds)})")
    else:
        print(f"  → Transcribiendo con faster-whisper '{model_name}'...")

    t0 = time.time()
    segments_gen, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5,
    )
    total_audio = info.duration if info.duration else (duration_s or 0)

    bar = tqdm(
        total=int(total_audio) if total_audio else None,
        unit="s", unit_scale=True, desc="    Whisper",
        bar_format="{l_bar}{bar}| {n:.0f}/{total:.0f}s [{elapsed}<{remaining}]" if total_audio else "{l_bar}{bar}| {elapsed}",
        dynamic_ncols=True,
    )

    entries, last_end = [], 0.0
    for seg in segments_gen:
        entries.append({
            "Text":    seg.text.strip(),
            "StartMs": int(seg.start * 1000),
            "EndMs":   int(seg.end   * 1000),
        })
        advance = seg.end - last_end
        if advance > 0:
            bar.update(int(advance))
            last_end = seg.end

    bar.close()
    elapsed = time.time() - t0
    print(f"  ✓ Transcripción completada en {fmt_seconds(elapsed)}  ({len(entries)} segmentos)")
    return entries


def save_cache(vod_id: str, entries: list) -> Path:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"{vod_id}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(entries, f, indent=2, ensure_ascii=False)
    return path


def upload(vod_id: str, entries: list) -> dict:
    try:
        resp = requests.post(
            UPLOAD_URL,
            json={"video_id": vod_id, "entries": entries},
            timeout=30,
        )
        return {"code": resp.status_code, "body": resp.json()}
    except requests.exceptions.ConnectionError:
        return {"code": None, "body": {"error": "No se pudo conectar al servidor"}}
    except Exception as e:
        return {"code": None, "body": {"error": str(e)}}


def process_vod(vod_id: str, model, model_name: str, duration_s=None, use_gpu: bool = False) -> bool:
    print(f"\n{'─'*55}")
    title_str = f"[VOD {vod_id}]"
    if duration_s:
        title_str += f"  duración: {fmt_seconds(duration_s)}"
    print(title_str)

    vod_start = time.time()
    cached = CACHE_DIR / f"{vod_id}.json"

    if cached.exists():
        print(f"  → Transcript cacheado encontrado, saltando descarga.")
        with open(cached, encoding="utf-8") as f:
            entries = json.load(f)
    else:
        tmp_dir = Path(tempfile.mkdtemp(prefix=f"vod_{vod_id}_"))
        audio_base = tmp_dir / vod_id
        try:
            if not download_audio(vod_id, audio_base, duration_s):
                return False
            audio_file = find_audio_file(audio_base)
            if not audio_file:
                print("  ✗ No se encontró el archivo de audio descargado")
                return False
            entries = transcribe(audio_file, model, model_name, duration_s, use_gpu)
            if not entries:
                print("  ✗ Whisper no generó segmentos")
                return False
            path = save_cache(vod_id, entries)
            print(f"  → Guardado en cache: {path}")
        except Exception as e:
            print(f"  ✗ Error inesperado: {e}")
            return False
        finally:
            shutil.rmtree(tmp_dir, ignore_errors=True)

    print(f"  → Subiendo {len(entries)} segmentos al backend...")
    res = upload(vod_id, entries)
    code, body = res["code"], res["body"]
    total = fmt_seconds(time.time() - vod_start)

    if code == 200:
        print(f"  ✓ Actualizado — {body.get('entries_saved', len(entries))} entradas  |  total: {total}")
        return True
    elif code == 201:
        print(f"  ✓ Creado     — {body.get('entries_saved', len(entries))} entradas  |  total: {total}")
        return True
    elif code == 404:
        print(f"  ✗ VOD no existe en la BD: {body.get('error', '')}")
        return False
    else:
        print(f"  ✗ Error [{code}]: {body}")
        return False


def process_all(model, model_name: str, use_gpu: bool = False):
    print("[INFO] Obteniendo lista de VODs desde la API...")
    vods = get_all_vods()
    if not vods:
        print("[ERROR] No hay VODs disponibles.")
        return

    speed_table = WHISPER_SPEED_GPU if use_gpu else WHISPER_SPEED_CPU
    speed_x = speed_table.get(model_name, 20.0 if use_gpu else 5.0)
    total_duration = sum(v.get("length_seconds") or 0 for v in vods)
    est_download   = total_duration / DOWNLOAD_SPEED_X
    est_whisper    = total_duration / speed_x

    print(f"[INFO] {len(vods)} VODs  |  duración total: {fmt_seconds(total_duration)}")
    print(f"       Estimado descarga:  ~{fmt_seconds(est_download)}")
    print(f"       Estimado Whisper ({model_name}  {'GPU' if use_gpu else 'CPU'}): ~{fmt_seconds(est_whisper)}")
    print(f"       Estimado total:    ~{fmt_seconds(est_download + est_whisper)}\n")

    global_start = time.time()
    success, failed = [], []

    for vod in vods:
        vod_id = vod["id"]
        duration_s = vod.get("length_seconds")
        ok = process_vod(vod_id, model, model_name, duration_s, use_gpu)
        (success if ok else failed).append(vod_id)

    elapsed_total = time.time() - global_start
    print(f"\n{'═'*55}")
    print(f"Completado en {fmt_seconds(elapsed_total)}: {len(success)} exitosos  |  {len(failed)} fallidos")
    if failed:
        print(f"Fallidos: {failed}")


print("✓ Funciones cargadas")

In [ ]:
# Celda 5 — Cargar modelo
from faster_whisper import WhisperModel
compute_type = "float16" if torch.cuda.is_available() else "int8"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = WhisperModel(MODEL_NAME, device=device, compute_type=compute_type)
print(f"Modelo '{MODEL_NAME}' listo en {device} ({compute_type})")

In [ ]:
# Celda 6 — Ejecutar (un VOD)
info = get_video_info(VOD_ID)
duration_s = info.get("length_seconds") if info else None
process_vod(VOD_ID, model, MODEL_NAME, duration_s, use_gpu=(device == "cuda"))

In [ ]:
# Celda 7 — Ejecutar todos los VODs (opcional)
# Descarga lista de VODs del backend y procesa todos
process_all(model, MODEL_NAME, use_gpu=(device == "cuda"))